In [6]:
import os
os.environ["HF_TOKEN"] = "REDACTED"

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SENTIMENT_MODEL = "mdhugol/indonesia-bert-sentiment-classification"
EMOTION_MODEL = "StevenLimcorn/indonesian-roberta-base-emotion-classifier"

SENTIMENT_LABELS = {0: "positive", 1: "neutral", 2: "negative"}
EMOTION_LABELS = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}

TEST_SENTENCES = [
    "Saya sangat senang dengan produk ini, kualitasnya bagus sekali!",
    "Produk ini buruk, kecewa banget dengan kualitasnya.",
    "Ini produk biasa saja, tidak istimewa.",
    "Aku takut akan terjadi hal buruk di masa depan.",
    "Saya marah karena layanan pelanggan sangat jelek.",
    "Cinta adalah hal terindah di dunia.",
]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n{'='*80}")
print(f"ML Model Tester - Download & Test Sentiment + Emotion Models")
print(f"Device: {device.upper()}\n")


🚀 ML Model Tester - Download & Test Sentiment + Emotion Models
Device: CPU



In [ ]:
# Load Sentiment
print("Loading Sentiment Model...")
sent_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
sent_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL)
sent_model.to(device).eval()
print("Sentiment model loaded!\n")

📥 Loading Sentiment Model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11323.17it/s]
BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Sentiment model loaded!



In [ ]:
# Load Emotion
print("Loading Emotion Model...")
emo_tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL)
emo_model = AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL)
emo_model.to(device).eval()
print("Emotion model loaded!\n")

📥 Loading Emotion Model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2131.26it/s]
RobertaForSequenceClassification LOAD REPORT from: StevenLimcorn/indonesian-roberta-base-emotion-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Emotion model loaded!



In [9]:
# Test
print(f"{'='*80}")
print(f"TESTING ON {len(TEST_SENTENCES)} SENTENCES\n")

TESTING ON 6 SENTENCES



In [10]:
for idx, text in enumerate(TEST_SENTENCES, 1):
    # Sentiment
    inputs_sent = sent_tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs_sent = {k: v.to(device) for k, v in inputs_sent.items()}
    with torch.no_grad():
        out_sent = sent_model(**inputs_sent)
        probs_sent = torch.softmax(out_sent.logits, dim=-1)[0]
    sent_label = SENTIMENT_LABELS[int(torch.argmax(probs_sent))]
    sent_score = float(torch.max(probs_sent))
    
    # Emotion
    inputs_emo = emo_tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs_emo = {k: v.to(device) for k, v in inputs_emo.items()}
    with torch.no_grad():
        out_emo = emo_model(**inputs_emo)
        probs_emo = torch.softmax(out_emo.logits, dim=-1)[0]
    emo_label = EMOTION_LABELS[int(torch.argmax(probs_emo))]
    emo_score = float(torch.max(probs_emo))
    
    # Print
    print(f"#{idx}: {text[:65]}...")
    print(f"   Sentiment: {sent_label.upper()} ({sent_score:.2%})")
    print(f"   Emotion: {emo_label.upper()} ({emo_score:.2%})\n")

#1: Saya sangat senang dengan produk ini, kualitasnya bagus sekali!...
   Sentiment: POSITIVE (99.63%)
   Emotion: FEAR (95.16%)

#2: Produk ini buruk, kecewa banget dengan kualitasnya....
   Sentiment: NEGATIVE (99.81%)
   Emotion: SADNESS (78.42%)

#3: Ini produk biasa saja, tidak istimewa....
   Sentiment: NEGATIVE (99.74%)
   Emotion: SADNESS (47.60%)

#4: Aku takut akan terjadi hal buruk di masa depan....
   Sentiment: NEGATIVE (99.68%)
   Emotion: ANGER (95.15%)

#5: Saya marah karena layanan pelanggan sangat jelek....
   Sentiment: NEGATIVE (99.84%)
   Emotion: JOY (84.88%)

#6: Cinta adalah hal terindah di dunia....
   Sentiment: POSITIVE (98.74%)
   Emotion: LOVE (97.77%)

